# Exercise 3 — market_drawdown and apply_drawdown_limit

`market_drawdown` computes the running distance below the rolling peak — the same formula as Day 91's `max_drawdown` but applied to a price series rather than an equity curve, returning a Series rather than a scalar. `apply_drawdown_limit` uses it as a regime filter: when the market is in a deep drawdown, all signals go flat.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def kelly_fraction(win_rate, avg_win, avg_loss):
    if avg_loss <= 0 or win_rate <= 0 or win_rate >= 1:
        return 0.0
    b = avg_win / avg_loss
    return max(0.0, min(1.0, win_rate - (1 - win_rate) / b))
def is_stopped_out(entry_price, current_price, stop_pct=0.05):
    if entry_price <= 0:
        return False
    return current_price <= entry_price * (1.0 - stop_pct)
def apply_stop_loss(signals, prices, stop_pct=0.05):
    result = signals.copy().astype(float)
    entry_price = None
    for i in range(len(result)):
        if result.iloc[i] == 1:
            if entry_price is None:
                entry_price = float(prices.iloc[i])
            elif is_stopped_out(entry_price, float(prices.iloc[i]), stop_pct):
                result.iloc[i] = 0
                entry_price = None
        else:
            entry_price = None
    return result.astype(int)

def market_drawdown(prices):
    """Running drawdown from the rolling peak at each bar.

    drawdown[i] = (prices[i] − peak[i]) / peak[i]
    where peak[i] = max(prices[0 … i])

    Implementation:
        peak = prices.cummax()
        return (prices - peak) / peak

    Returns pd.Series of values ≤ 0.
    """
    # TODO: two lines
    return pd.Series(0.0, index=prices.index)


def apply_drawdown_limit(signals, prices, limit=-0.20):
    """Zero signals when the market drawdown exceeds `limit`.

    Implementation:
        dd     = market_drawdown(prices)
        result = signals.copy().astype(int)
        result[dd < limit] = 0
        return result

    Args:
        limit : float ≤ 0 — e.g. -0.20 means halt when down 20% from peak
    """
    # TODO: three lines
    return signals.copy().astype(int)


### Checks

In [ ]:
checks = 0

# 1 — market_drawdown: monotone rising → all zeros
try:
    dates  = pd.date_range("2023-01-01", periods=10, freq="B")
    rising = pd.Series([float(100 + i) for i in range(10)], index=dates)
    dd     = market_drawdown(rising)
    assert isinstance(dd, pd.Series) and len(dd) == 10
    assert dd.abs().max() < 1e-9, f"rising prices → dd = 0, got max {dd.abs().max()}"
    checks += 1; print("✅ 1 market_drawdown: monotone rising → all zeros")
except Exception as e:
    print("❌ 1:", e)

# 2 — market_drawdown: known 50% drawdown
try:
    dates = pd.date_range("2023-01-01", periods=3, freq="B")
    p     = pd.Series([100.0, 150.0, 75.0], index=dates)
    dd    = market_drawdown(p)
    assert abs(dd.iloc[0]) < 1e-9, f"bar 0: expected 0, got {dd.iloc[0]}"
    assert abs(dd.iloc[1]) < 1e-9, f"bar 1: expected 0, got {dd.iloc[1]}"
    assert abs(dd.iloc[2] - (-0.5)) < 1e-9, f"bar 2: expected -0.5, got {dd.iloc[2]}"
    checks += 1; print("✅ 2 market_drawdown: 100→150→75 gives dd=[0, 0, -0.5]")
except Exception as e:
    print("❌ 2:", e)

# 3 — apply_drawdown_limit: controlled 5-bar test
try:
    dates  = pd.date_range("2023-01-01", periods=5, freq="B")
    prices = pd.Series([100.0, 110.0, 120.0, 90.0, 80.0], index=dates)
    sig    = pd.Series([1, 1, 1, 1, 1], index=dates)
    res    = apply_drawdown_limit(sig, prices, limit=-0.20)
    # bar 3: dd=(90-120)/120=-0.25 < -0.20 → 0
    # bar 4: dd=(80-120)/120=-0.33 < -0.20 → 0
    assert res.tolist() == [1, 1, 1, 0, 0],         f"expected [1,1,1,0,0], got {res.tolist()}"
    checks += 1; print("✅ 3 apply_drawdown_limit: bars 3 and 4 zeroed (dd < -20%)")
except Exception as e:
    print("❌ 3:", e)

# 4 — apply_drawdown_limit: tight limit zeroes out most bars on sine-wave
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    res = apply_drawdown_limit(sig, df["Close"], limit=-0.10)
    assert res.sum() < sig.sum(),         "tight limit should zero some bars on sine-wave data"
    checks += 1; print("✅ 4 tight limit reduces long exposure on sine-wave data")
except Exception as e:
    print("❌ 4:", e)

# 5 — apply_drawdown_limit: very loose limit keeps all bars
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    res = apply_drawdown_limit(sig, df["Close"], limit=-1.00)  # -100%: never triggered
    assert (res == 1).all(), "limit=-1.0 should never trigger → all 1s"
    checks += 1; print("✅ 5 limit=-1.0 (never triggers) → all signals unchanged")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
